In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# ============================================================
# CARGA DESDE GOLD SEMANTIC LAYER - ÚLTIMO data_ingestion_ts
# ============================================================

print("=" * 60)
print("CARGA DESDE GOLD SEMANTIC LAYER")
print("=" * 60)

table_semantic = "adbsmartdatamanuelestrada.gold.semantic_layer_gld"

# 1. Obtener el último timestamp de ingestión
ultimo_ts = spark.table(table_semantic) \
    .select(F.max("data_ingestion_ts").alias("max_ts")) \
    .collect()[0]["max_ts"]

print(f"\nÚltimo data_ingestion_ts en Gold Semantic: {ultimo_ts}")

# 2. Cargar solo los datos del último timestamp
df_semantic = spark.table(table_semantic) \
    .filter(F.col("data_ingestion_ts") == ultimo_ts)

registros_cargados = df_semantic.count()

print(f"\nRegistros cargados: {registros_cargados:,}")
print(f"Columnas: {len(df_semantic.columns)}")
print("\nEsquema:")
df_semantic.printSchema()
print("=" * 60)

In [0]:
# ============================================================
# AGREGACIÓN POR PRODUCTO, TIENDA Y PERIODO MENSUAL
# ============================================================

print("=" * 60)
print("AGREGACIÓN POR PRODUCTO, TIENDA Y PERIODO MENSUAL")
print("=" * 60)

print("\nCreando dimensiones de periodo (año/mes)...")

# 1. Crear columna periodo como primer día del mes (tipo date)
df_con_periodo = df_semantic \
    .withColumn("periodo", F.trunc("fecha_orden", "MM"))

print("\nAgregando métricas por cod_producto, cod_tienda y periodo...")

# 2. Agrupar por cod_producto, cod_tienda y periodo, sumando las métricas
df_analitico = df_con_periodo \
    .groupBy("cod_producto", "cod_tienda", "periodo") \
    .agg(
        F.sum("cantidad").alias("total_cantidad"),
        F.sum("precio_unit").alias("total_precio_unit"),
        F.sum("coste_unit").alias("total_coste_unit"),
        F.sum("venta").alias("total_venta"),
        F.sum("coste_venta").alias("total_coste_venta"),
        F.sum("iva").alias("total_iva"),
        F.sum("ventasiniva").alias("total_ventasiniva"),
        F.sum("margen").alias("total_margen"),
        F.count("*").alias("num_transacciones")
    ) \
    .withColumn("data_ingestion_ts", F.lit(ultimo_ts)) \
    .orderBy("periodo", "cod_tienda", "cod_producto")

registros_agregados = df_analitico.count()

print(f"\n✅ Agregación completada")
print(f"Registros originales: {registros_cargados:,}")
print(f"Registros agregados: {registros_agregados:,}")

print("\nVista previa de datos agregados:")
display(df_analitico.limit(10))

print("\nResumen por periodo:")
df_analitico.groupBy("periodo") \
    .agg(
        F.count("cod_producto").alias("productos_distintos"),
        F.sum("num_transacciones").alias("total_transacciones"),
        F.sum("total_venta").alias("venta_periodo"),
        F.sum("total_margen").alias("margen_periodo")
    ) \
    .orderBy("periodo") \
    .show()

print("=" * 60)

In [0]:
# ============================================================
# CONTROL DE DUPLICADOS POR PERIODO - GOLD ANALÍTICA
# ============================================================

print("=" * 60)
print("CONTROL DE DUPLICADOS POR PERIODO")
print("=" * 60)

table_analitica = "adbsmartdatamanuelestrada.gold.flujo_analitico_mensual_gld"

# Obtener los periodos en el DataFrame actual
periodos_en_datos = df_analitico.select("periodo").distinct().collect()
periodos_carga = [row['periodo'] for row in periodos_en_datos]
print(f"\nPeriodo(s) en los datos a cargar: {[str(p) for p in periodos_carga]}")

if len(periodos_carga) > 1:
    raise ValueError(
        f"❌ ERROR: Los datos contienen múltiples periodos: {[str(p) for p in periodos_carga]}\n"
        f"   La carga incremental debe ser de un solo periodo a la vez."
    )

periodo_actual = periodos_carga[0]
print(f"Periodo de la carga: {periodo_actual}")

# Verificar si el periodo ya existe en la tabla Gold Analítica
try:
    # Intentar leer la tabla (si no existe, se creará con overwrite)
    periodos_existentes = spark.table(table_analitica) \
        .select("periodo") \
        .distinct() \
        .collect()
    
    periodos_existentes_list = [row['periodo'] for row in periodos_existentes]
    print(f"\nPeriodos ya existentes en tabla Gold Analítica: {[str(p) for p in periodos_existentes_list]}")
    
    # Verificar si el periodo actual ya existe
    if periodo_actual in periodos_existentes_list:
        puede_appendear = False
        print(f"\n❌ El periodo {periodo_actual} YA EXISTE en la tabla.")
        print("   No se puede appendear datos duplicados.")
        print("   Carga BLOQUEADA para evitar duplicación.")
    else:
        puede_appendear = True
        print(f"\n✅ El periodo {periodo_actual} NO existe en la tabla. Se procederá a appendear.")
        
except Exception as e:
    # Si la tabla no existe, será la primera carga (overwrite)
    puede_appendear = True
    print(f"\nTabla no existe. Primera carga → se creará con OVERWRITE.")

print("\n" + "=" * 60)

In [0]:
# ============================================================
# ESCRITURA A TABLA GOLD ANALÍTICA
# ============================================================

print("=" * 60)
print("ESCRITURA A TABLA GOLD ANALÍTICA")
print("=" * 60)

print(f"\nTabla destino: {table_analitica}")

# Escribir a la tabla Gold Analítica según validación
if puede_appendear:
    print(f"Registros a insertar: {registros_agregados:,}")
    print(f"Columnas: {len(df_analitico.columns)}")
    
    print("\nEsquema del DataFrame analítico:")
    df_analitico.printSchema()
    
    # Determinar el modo de escritura
    try:
        # Verificar si la tabla existe
        spark.table(table_analitica).limit(1).collect()
        modo_escritura = "append"
        print(f"\nModo de escritura: APPEND (incremental)")
    except:
        modo_escritura = "overwrite"
        print(f"\nModo de escritura: OVERWRITE (primera carga)")
    
    # Escribir a la tabla
    df_analitico.write \
        .format("delta") \
        .mode(modo_escritura) \
        .option("overwriteSchema", "true") \
        .saveAsTable(table_analitica)
    
    print("\n" + "=" * 60)
    print("✅ INSERCIÓN EXITOSA")
    print("=" * 60)
    print(f"Tabla: {table_analitica}")
    print(f"Periodo insertado: {periodo_actual}")
    print(f"Registros insertados: {registros_agregados:,}")
    
    # Verificar la escritura
    total_registros = spark.table(table_analitica).count()
    print(f"Total registros en tabla: {total_registros:,}")
    print(f"Modo: {modo_escritura.upper()}")
    
else:
    print("\n" + "=" * 60)
    print("❌ CARGA CANCELADA")
    print("=" * 60)
    print(f"Razón: El periodo {periodo_actual} ya existe en la tabla")
    print(f"No se realizaron cambios en {table_analitica}")

print("=" * 60)